In [ ]:
# -*- coding: utf-8 -*-
"""
Plot-only seasonal climatology figures from saved GeoTIFFs
----------------------------------------------------------
Creates publication-style figure in projected CRS (EPSG:25832),
but shows axis tick labels as Longitude/Latitude.

Map geometry stays projected.
Only tick labels are converted to lon/lat.
"""

from pathlib import Path
import numpy as np
import geopandas as gpd
import matplotlib.pyplot as plt
import rasterio

from rasterio.warp import calculate_default_transform, reproject, Resampling
from rasterio.transform import array_bounds
from matplotlib.colors import Normalize
from matplotlib.ticker import MaxNLocator
from matplotlib.gridspec import GridSpec
from matplotlib.patches import Rectangle
from pyproj import Transformer


# =============================================================================
# PATHS
# =============================================================================
BASE = Path(r"C:\goyal_shekhar\post_doc\ecostress\predictions_rf_full_pubready_projected")

SEASON_TIFS = {
    "DJF": BASE / "Tw_pred_mean_DJF.tif",
    "MAM": BASE / "Tw_pred_mean_MAM.tif",
    "JJA": BASE / "Tw_pred_mean_JJA.tif",
    "SON": BASE / "Tw_pred_mean_SON.tif",
}

BASIN_SHP = Path(r"C:\goyal_shekhar\post_doc\ecostress\data\bode_shp\bode.shp")
RIVER_SHP = Path(r"C:\goyal_shekhar\post_doc\ecostress\data\HydroRIVERS_v10_eu_shp\HydroRIVERS_v10_eu_shp\HydroRIVERS_v10_eu.shp")

OUT_PNG = BASE / "Tw_pred_seasonal_climatology_projected_with_latlon_ticks.png"
OUT_PDF = BASE / "Tw_pred_seasonal_climatology_projected_with_latlon_ticks.pdf"


# =============================================================================
# SETTINGS
# =============================================================================
CMAP = "coolwarm"
PCTL_VMIN = 2
PCTL_VMAX = 98
SCALEBAR_KM = 10
PLOT_RIVERS = False

PROJECTED_CRS = "EPSG:25832"
GEOGRAPHIC_CRS = "EPSG:4326"

plt.rcParams.update({
    "font.family": "sans-serif",
    "font.sans-serif": ["Arial", "DejaVu Sans", "Liberation Sans"],
    "font.size": 10,
    "axes.labelsize": 10,
    "axes.titlesize": 12,
    "xtick.labelsize": 9,
    "ytick.labelsize": 9,
    "axes.linewidth": 0.8,
    "xtick.major.width": 0.8,
    "ytick.major.width": 0.8,
    "xtick.major.size": 3.0,
    "ytick.major.size": 3.0,
    "pdf.fonttype": 42,
    "ps.fonttype": 42,
    "figure.facecolor": "white",
    "axes.facecolor": "white",
    "savefig.facecolor": "white",
})


# =============================================================================
# HELPERS
# =============================================================================
def extent_from_transform(transform, width, height):
    left, bottom, right, top = array_bounds(height, width, transform)
    return [left, right, bottom, top]


def reproject_raster_to_crs(arr, src_transform, src_crs, dst_crs):
    h, w = arr.shape
    left, bottom, right, top = array_bounds(h, w, src_transform)

    dst_transform, dst_width, dst_height = calculate_default_transform(
        src_crs, dst_crs, w, h, left, bottom, right, top
    )

    dst = np.full((dst_height, dst_width), np.nan, dtype=np.float32)

    reproject(
        source=arr.astype(np.float32),
        destination=dst,
        src_transform=src_transform,
        src_crs=src_crs,
        src_nodata=np.nan,
        dst_transform=dst_transform,
        dst_crs=dst_crs,
        dst_nodata=np.nan,
        resampling=Resampling.bilinear
    )
    return dst, dst_transform


def add_north_arrow(ax, x=0.93, y=0.87, size=0.08):
    ax.annotate(
        "N",
        xy=(x, y), xycoords="axes fraction",
        xytext=(x, y - size), textcoords="axes fraction",
        ha="center", va="center",
        fontsize=11, fontweight="bold",
        arrowprops=dict(arrowstyle="-|>", lw=1.0, color="black")
    )


def add_scalebar_projected(ax, length_km=10, loc=(0.07, 0.06), height_frac=0.012, ndiv=2):
    xmin, xmax = ax.get_xlim()
    ymin, ymax = ax.get_ylim()

    x0 = xmin + loc[0] * (xmax - xmin)
    y0 = ymin + loc[1] * (ymax - ymin)
    bar_h = height_frac * (ymax - ymin)
    length_m = length_km * 1000.0
    seg = length_m / ndiv

    for i in range(ndiv):
        face = "black" if i % 2 == 0 else "white"
        ax.add_patch(Rectangle(
            (x0 + i * seg, y0), seg, bar_h,
            facecolor=face, edgecolor="black", lw=0.7, zorder=30
        ))

    ax.text(
        x0 + length_m / 2, y0 + 1.6 * bar_h, f"{length_km} km",
        ha="center", va="bottom", fontsize=9, zorder=31
    )


def zoom_to_basin(ax, basin_gdf, pad_frac=0.05):
    bminx, bminy, bmaxx, bmaxy = basin_gdf.total_bounds
    padx = pad_frac * (bmaxx - bminx)
    pady = pad_frac * (bmaxy - bminy)
    ax.set_xlim(bminx - padx, bmaxx + padx)
    ax.set_ylim(bminy - pady, bmaxy + pady)


def load_and_reproject_seasonal_rasters(dst_crs):
    arrays = {}
    transforms = {}
    all_vals = []

    for season, tif in SEASON_TIFS.items():
        if not tif.exists():
            raise FileNotFoundError(f"Missing raster: {tif}")

        with rasterio.open(tif) as src:
            arr = src.read(1).astype(np.float32)
            nodata = src.nodata
            if nodata is not None:
                arr[arr == nodata] = np.nan

            arr_p, transform_p = reproject_raster_to_crs(
                arr, src.transform, src.crs, dst_crs
            )
            arrays[season] = arr_p
            transforms[season] = transform_p

            vals = arr_p[np.isfinite(arr_p)]
            if vals.size > 0:
                all_vals.append(vals)

    if len(all_vals) == 0:
        raise RuntimeError("No finite raster values found in seasonal GeoTIFFs.")

    all_vals = np.concatenate(all_vals)
    vmin = float(np.nanpercentile(all_vals, PCTL_VMIN))
    vmax = float(np.nanpercentile(all_vals, PCTL_VMAX))

    return arrays, transforms, vmin, vmax


def set_latlon_ticks_on_projected_axes(ax, src_crs=PROJECTED_CRS, dst_crs=GEOGRAPHIC_CRS, n=5):
    """
    Keep the axes in projected CRS, but relabel ticks as lon/lat.
    This is appropriate here because the basin is spatially small.
    """
    transformer = Transformer.from_crs(src_crs, dst_crs, always_xy=True)

    xmin, xmax = ax.get_xlim()
    ymin, ymax = ax.get_ylim()

    # choose clean projected tick positions
    xticks = MaxNLocator(nbins=n).tick_values(xmin, xmax)
    yticks = MaxNLocator(nbins=n).tick_values(ymin, ymax)

    # retain only visible ticks
    xticks = xticks[(xticks >= xmin) & (xticks <= xmax)]
    yticks = yticks[(yticks >= ymin) & (yticks <= ymax)]

    # Use bottom axis line for longitude labels
    lon_labels = []
    y_ref = ymin
    for x in xticks:
        lon, _ = transformer.transform(x, y_ref)
        lon_labels.append(f"{lon:.2f}°E")

    # Use left axis line for latitude labels
    lat_labels = []
    x_ref = xmin
    for y in yticks:
        _, lat = transformer.transform(x_ref, y)
        lat_labels.append(f"{lat:.2f}°N")

    ax.set_xticks(xticks)
    ax.set_yticks(yticks)
    ax.set_xticklabels(lon_labels)
    ax.set_yticklabels(lat_labels)


def make_seasonal_figure(dst_crs, out_png, out_pdf, title):
    arrays, transforms, vmin, vmax = load_and_reproject_seasonal_rasters(dst_crs)

    basin = gpd.read_file(BASIN_SHP).to_crs(dst_crs)
    river = gpd.read_file(RIVER_SHP).to_crs(dst_crs)

    try:
        river = gpd.clip(river, basin)
    except Exception:
        pass

    order = ["DJF", "MAM", "JJA", "SON"]
    panel_labels = ["(a)", "(b)", "(c)", "(d)"]

    fig = plt.figure(figsize=(11.8, 8.4), facecolor="white")
    gs = GridSpec(
        2, 3,
        width_ratios=[1, 1, 0.055],
        height_ratios=[1, 1],
        wspace=0.16,
        hspace=0.22
    )

    axes = [
        fig.add_subplot(gs[0, 0]),
        fig.add_subplot(gs[0, 1]),
        fig.add_subplot(gs[1, 0]),
        fig.add_subplot(gs[1, 1]),
    ]
    cax = fig.add_subplot(gs[:, 2])

    last_im = None

    for i, season in enumerate(order):
        ax = axes[i]
        ax.set_facecolor("white")

        arr = arrays[season]
        transform = transforms[season]
        ext = extent_from_transform(transform, arr.shape[1], arr.shape[0])

        basin.plot(ax=ax, facecolor="#f5f5f5", edgecolor="none", zorder=0)

        if np.isfinite(arr).any():
            last_im = ax.imshow(
                np.ma.masked_invalid(arr),
                origin="upper",
                extent=ext,
                interpolation="nearest",
                cmap=CMAP,
                norm=Normalize(vmin=vmin, vmax=vmax),
                zorder=2
            )
        else:
            ax.text(
                0.5, 0.5, "No data",
                transform=ax.transAxes,
                ha="center", va="center",
                fontsize=11, fontweight="bold", color="0.4"
            )

        if PLOT_RIVERS and not river.empty:
            river.plot(ax=ax, linewidth=0.3, color="0.80", alpha=0.8, zorder=1)

        basin.boundary.plot(ax=ax, linewidth=1.0, color="black", alpha=0.95, zorder=5)

        zoom_to_basin(ax, basin, pad_frac=0.05)
        ax.set_aspect("equal", adjustable="box")

        # projected map, but lon/lat tick labels
        set_latlon_ticks_on_projected_axes(ax, src_crs=PROJECTED_CRS, dst_crs=GEOGRAPHIC_CRS, n=5)

        ax.set_title(f"{panel_labels[i]} {season}", fontweight="bold", pad=6)
        ax.grid(False)
        ax.tick_params(direction="out", length=3.0, width=0.8)

        if i in [2, 3]:
            ax.set_xlabel("Longitude")
        else:
            ax.set_xlabel("")

        if i in [0, 2]:
            ax.set_ylabel("Latitude")
        else:
            ax.set_ylabel("")

        for sp in ["top", "right"]:
            ax.spines[sp].set_visible(False)

    cbar = fig.colorbar(last_im, cax=cax)
    cbar.set_label("Predicted river water temperature (°C)", fontsize=11)
    cbar.ax.tick_params(labelsize=9)
    cbar.locator = MaxNLocator(nbins=6)
    cbar.update_ticks()

    add_north_arrow(axes[0], x=0.93, y=0.87, size=0.08)
    add_scalebar_projected(axes[0], length_km=SCALEBAR_KM, ndiv=2, loc=(0.07, 0.06))

    fig.suptitle(title, y=0.98, fontsize=16, fontweight="bold")

    
    plt.show()
    plt.close(fig)

    print("Saved:", out_png)
    print("Saved:", out_pdf)


# =============================================================================
# RUN
# =============================================================================
make_seasonal_figure(
    dst_crs=PROJECTED_CRS,
    out_png=OUT_PNG,
    out_pdf=OUT_PDF,
    title="Seasonal mean predicted river water temperature"
)